In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

FILE_LOCATION = '/kaggle/input/competitions/playground-series-s6e8/'
train_dataset = pd.read_csv(FILE_LOCATION + 'train.csv')
test_dataset = pd.read_csv(FILE_LOCATION + 'test.csv')
TARGET = train_dataset['addicted_label']
train_dataset = train_dataset.drop(columns=['addicted_label', 'id','gender', 'academic_work_impact','age', 'stress_level'])
y_id = test_dataset['id']
X_test = test_dataset.drop(columns=['id','gender', 'academic_work_impact','age', 'stress_level'])
N_SPLITS = 10
del test_dataset

/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e8/train.csv
/kaggle/input/competitions/playground-series-s6e8/test.csv


In [2]:
%pip install xgboost
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
from xgboost import XGBClassifier

use_device = 'cuda' if torch.cuda.is_available() else 'cpu'

TE_COLS = train_dataset.columns.tolist()
SMOOTHING = 10.0
best_params = {'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 0.1, 'min_child_weight': 7,
                'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


def to_levels(df, cols):
    """Convert columns to string labels for grouping. NaN becomes an
    explicit '__missing__' level, safe across pandas versions."""
    return pd.DataFrame({
        c: df[c].astype(object).fillna("__missing__").astype(str).values
        for c in cols
    }, index=df.index)


def build_te_maps(levels_df, y_arr, cols, smoothing=SMOOTHING):
    """Smoothed target-mean map + frequency map per column, built ONLY
    from the rows passed in."""
    global_mean = y_arr.mean()
    maps = {}
    for c in cols:
        grouped = pd.DataFrame({"lv": levels_df[c].values, "y": y_arr}) \
                    .groupby("lv")["y"].agg(["count", "mean"])
        smoothed = (grouped["count"] * grouped["mean"] + smoothing * global_mean) / \
                   (grouped["count"] + smoothing)
        maps[c] = (smoothed.astype(np.float32), grouped["count"].astype(np.float32))
    return maps, global_mean


def apply_te_maps(levels_df, cols, maps, global_mean):
    """Apply previously-built maps to produce te_ and fq_ columns
    (interleaved: te_c1, fq_c1, te_c2, fq_c2, ...)."""
    out = {}
    for c in cols:
        te_map, fq_map = maps[c]
        out[f"te_{c}"] = levels_df[c].map(te_map).astype(np.float32).fillna(global_mean).values
        out[f"fq_{c}"] = levels_df[c].map(fq_map).astype(np.float32).fillna(0.0).values
    return pd.DataFrame(out, index=levels_df.index)


def build_oof_encoding(X_tr, y_tr, cols, n_inner=5, smoothing=SMOOTHING, random_state=48):
    """Leak-free encoding for the training rows themselves: each row's
    encoded value comes from an inner fold that excludes that row."""
    levels_tr = to_levels(X_tr, cols)
    te_fq_cols = [item for c in cols for item in (f"te_{c}", f"fq_{c}")]  # interleaved,
                                                                            # matches apply_te_maps
    oof = pd.DataFrame(
        np.zeros((len(X_tr), len(te_fq_cols)), dtype=np.float32),
        columns=te_fq_cols,
        index=X_tr.index
    )
    inner_cv = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=random_state)
    for inner_tr_idx, inner_ho_idx in inner_cv.split(X_tr, y_tr):
        maps, global_mean = build_te_maps(
            levels_tr.iloc[inner_tr_idx], y_tr.iloc[inner_tr_idx].values, cols, smoothing
        )
        encoded = apply_te_maps(levels_tr.iloc[inner_ho_idx], cols, maps, global_mean)
        # Explicitly reorder by column name before assigning, so a mismatch
        # between this function's column order and apply_te_maps' can't
        # silently scramble values again.
        oof.iloc[inner_ho_idx] = encoded[te_fq_cols].values.astype(np.float32)
    return oof


# ---- main CV loop ----
X_full = train_dataset
y_full = TARGET
obj_cols = X_full.select_dtypes(include='object').columns
X_full[obj_cols] = X_full[obj_cols].astype('category')

X_test = X_test[X_full.columns]          # force X_test's column order to match X_full's
X_test[obj_cols] = X_test[obj_cols].astype('category')
X_test_reset = X_test.reset_index(drop=True)

oof_predictions = np.zeros(len(X_full))

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=48)

fold_roc_aucs = []
fold_accuracies = []
test_probas_folds = []

for fold_num, (train_idx, val_idx) in enumerate(cv.split(X_full, y_full), start=1):
    X_tr = X_full.iloc[train_idx].reset_index(drop=True)
    X_va = X_full.iloc[val_idx].reset_index(drop=True)
    y_tr = y_full.iloc[train_idx].reset_index(drop=True)
    y_va = y_full.iloc[val_idx].reset_index(drop=True)

    # 1. X_tr gets its own leak-free out-of-fold encoding (inner 5-fold)
    tr_te = build_oof_encoding(X_tr, y_tr, TE_COLS, n_inner=5, random_state=48)

    # 2. One map from the WHOLE of X_tr, applied to X_va and X_test (safe: neither
    #    contributed to building it)
    full_maps, full_global_mean = build_te_maps(to_levels(X_tr, TE_COLS), y_tr.values, TE_COLS)
    va_te = apply_te_maps(to_levels(X_va, TE_COLS), TE_COLS, full_maps, full_global_mean)
    test_te = apply_te_maps(to_levels(X_test_reset, TE_COLS), TE_COLS, full_maps, full_global_mean)

    X_tr_final = pd.concat([X_tr, tr_te], axis=1)
    X_va_final = pd.concat([X_va, va_te.reset_index(drop=True)], axis=1)
    X_test_final = pd.concat([X_test_reset, test_te.reset_index(drop=True)], axis=1)

    model = XGBClassifier(
        missing=np.nan,
        enable_categorical=True,
        tree_method='hist',
        device=use_device,
        random_state=48,
        eval_metric='auc',
        n_estimators=5000,
        early_stopping_rounds=50,
        **best_params
    )
    model.fit(X_tr_final, y_tr, eval_set=[(X_va_final, y_va)], verbose=False)

    y_proba = model.predict_proba(X_va_final)[:, 1]
    oof_predictions[val_idx] = y_proba
    y_pred = model.predict(X_va_final)

    fold_auc = roc_auc_score(y_va, y_proba)
    fold_acc = accuracy_score(y_va, y_pred)
    fold_roc_aucs.append(fold_auc)
    fold_accuracies.append(fold_acc)
    test_probas_folds.append(model.predict_proba(X_test_final)[:, 1])

    print(f"Fold {fold_num}: ROC-AUC = {fold_auc:.6f}, Accuracy = {fold_acc:.6f}, best_iteration = {model.best_iteration}")

fold_roc_aucs = np.array(fold_roc_aucs)
fold_accuracies = np.array(fold_accuracies)

print(f"\nMean ROC-AUC: {fold_roc_aucs.mean():.6f}  |  Std: {fold_roc_aucs.std():.6f}")
print(f"Mean Accuracy: {fold_accuracies.mean():.6f}  |  Std: {fold_accuracies.std():.6f}")

Note: you may need to restart the kernel to use updated packages.
Fold 1: ROC-AUC = 0.966924, Accuracy = 0.905261, best_iteration = 1543
Fold 2: ROC-AUC = 0.965359, Accuracy = 0.903684, best_iteration = 1001
Fold 3: ROC-AUC = 0.966799, Accuracy = 0.905058, best_iteration = 1552
Fold 4: ROC-AUC = 0.964885, Accuracy = 0.902831, best_iteration = 1231
Fold 5: ROC-AUC = 0.965395, Accuracy = 0.905073, best_iteration = 1273
Fold 6: ROC-AUC = 0.965887, Accuracy = 0.903583, best_iteration = 1323
Fold 7: ROC-AUC = 0.966043, Accuracy = 0.904754, best_iteration = 1190
Fold 8: ROC-AUC = 0.967273, Accuracy = 0.906215, best_iteration = 1502
Fold 9: ROC-AUC = 0.966868, Accuracy = 0.905130, best_iteration = 1783
Fold 10: ROC-AUC = 0.966738, Accuracy = 0.906821, best_iteration = 1639

Mean ROC-AUC: 0.966217  |  Std: 0.000773
Mean Accuracy: 0.904841  |  Std: 0.001146


In [3]:
# Submission: average predictions across all N_SPLITS fold models
test_probabilities = np.mean(test_probas_folds, axis=0)

submission = pd.DataFrame({
    'id': y_id,
    'addicted_label': test_probabilities
})
submission.to_csv('xgboost_submission.csv', index=False)

oof_df = pd.DataFrame({
    'id': X_full['id'] if 'id' in X_full.columns else np.arange(len(X_full)),
    'true_label': y_full.values,
    'oof_proba': oof_predictions
})
oof_df.to_csv('xgboost_oof.csv', index=False)

test_probs_df = pd.DataFrame({
    'id': y_id,
    'test_proba': test_probabilities
})
test_probs_df.to_csv('xgboost_test_probs.csv', index=False)

print(f"Full OOF ROC-AUC: {roc_auc_score(y_full, oof_predictions):.6f}")
print(submission.head())

Full OOF ROC-AUC: 0.966219
       id  addicted_label
0  691369        0.999740
1  691370        0.959882
2  691371        0.657737
3  691372        0.987093
4  691373        0.995797
